In [ ]:
# Ячейка 1: Импорты и константы
import os
import cv2
import numpy as np
import pandas as pd
import albumentations as A
from albumentations.pytorch import ToTensorV2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

BASE_PATH = '/home/shizm/DL_LABs/DL-lab4/dl-lab-4-ocr/'
TEST_IMG_DIR = os.path.join(BASE_PATH, 'test', 'test')
SAMPLE_SUBMISSION = os.path.join(BASE_PATH, 'sample_submission.csv')
TARGET_HEIGHT = 32
BATCH_SIZE = 128
NUM_WORKERS = 4
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Устройство: {DEVICE}")

Устройство: cuda


In [42]:
# Ячейка 2: Алфавит и модель CRNN
ALPHABET = '0123456789'
CHAR_TO_IDX = {char: idx + 1 for idx, char in enumerate(ALPHABET)}
IDX_TO_CHAR = {idx + 1: char for idx, char in enumerate(ALPHABET)}
BLANK_IDX = 0
NUM_CLASSES = len(ALPHABET) + 1

class CRNN(nn.Module):
    def __init__(self, num_classes, input_channels=3, hidden_size=256):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(input_channels, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=(2, 1), stride=(2, 1)),
            nn.Conv2d(256, 512, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=(2, 1), stride=(2, 1)),
            nn.AdaptiveAvgPool2d((1, None))
        )
        self.cnn_output_channels = 512
        self.lstm = nn.LSTM(input_size=self.cnn_output_channels, hidden_size=hidden_size,
                            num_layers=2, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(hidden_size * 2, num_classes)
    
    def forward(self, x):
        conv_features = self.cnn(x)
        conv_features = conv_features.squeeze(2)
        conv_features = conv_features.permute(0, 2, 1)
        lstm_out, _ = self.lstm(conv_features)
        logits = self.fc(lstm_out)
        logits = logits.permute(1, 0, 2)
        return logits

In [43]:
# Ячейка 3: Декодер (проверенный greedy)
def decode_predictions(logits, blank_idx=BLANK_IDX):
    logits = logits.permute(1, 0, 2)
    _, max_indices = torch.max(logits, dim=2)
    decoded = []
    for indices in max_indices:
        chars = []
        prev = blank_idx
        for idx in indices:
            idx = idx.item()
            if idx != blank_idx and idx != prev:
                chars.append(IDX_TO_CHAR[idx])
            prev = idx
        decoded.append(''.join(chars))
    return decoded

In [44]:
# Ячейка 4: Трансформации и Dataset
test_transforms = A.Compose([
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

class TestDataset(Dataset):
    def __init__(self, df, img_dir, transforms=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transforms = transforms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row['Filename'])
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        h, w = image.shape[:2]
        new_w = int(w * (TARGET_HEIGHT / h))
        image = cv2.resize(image, (new_w, TARGET_HEIGHT), interpolation=cv2.INTER_LINEAR)
        if self.transforms:
            augmented = self.transforms(image=image)
            image = augmented['image']
        return image, row['Filename']

In [45]:
# Ячейка 5: TTA функция
def apply_tta(image_tensor, model):
    """
    Применяет TTA: оригинал, +3°, -3° и возвращает усреднённые логиты.
    """
    def rotate_tensor(x, angle):
        x_np = x.permute(1, 2, 0).cpu().numpy()
        rotated = A.rotate(x_np, angle=angle, border_mode=cv2.BORDER_CONSTANT, value=0)
        return torch.from_numpy(rotated).permute(2, 0, 1).to(DEVICE)
    
    logits_list = []
    with torch.no_grad():
        # Original
        logits_list.append(model(image_tensor.unsqueeze(0)))
        # +3 degrees
        logits_list.append(model(rotate_tensor(image_tensor, 3).unsqueeze(0)))
        # -3 degrees
        logits_list.append(model(rotate_tensor(image_tensor, -3).unsqueeze(0)))
    
    return torch.stack(logits_list).mean(dim=0)

In [46]:
# Ячейка 6: Загрузка моделей (используем _improved)
model_paths = [f'crnn_fold_{fold}_improved.pth' for fold in range(10)]

models = []
for path in model_paths:
    if os.path.exists(path):
        model = CRNN(num_classes=NUM_CLASSES).to(DEVICE)
        model.load_state_dict(torch.load(path, map_location=DEVICE))
        model.eval()
        models.append(model)
        print(f"Загружена модель: {path}")
    else:
        print(f"Файл не найден: {path}")

print(f"Всего загружено моделей: {len(models)}")

Загружена модель: crnn_fold_0_improved.pth
Загружена модель: crnn_fold_1_improved.pth


/tmp/ipykernel_220791/1698774855.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(path, map_location=DEVICE))


Загружена модель: crnn_fold_2_improved.pth
Загружена модель: crnn_fold_3_improved.pth
Загружена модель: crnn_fold_4_improved.pth
Загружена модель: crnn_fold_5_improved.pth
Загружена модель: crnn_fold_6_improved.pth
Загружена модель: crnn_fold_7_improved.pth
Загружена модель: crnn_fold_8_improved.pth
Загружена модель: crnn_fold_9_improved.pth
Всего загружено моделей: 10


In [47]:
# Ячейка 7: Инференс с голосованием (без TTA)
from collections import Counter

sub_df = pd.read_csv(SAMPLE_SUBMISSION)
test_dataset = TestDataset(sub_df, TEST_IMG_DIR, transforms=test_transforms)

def collate_test(batch):
    images, filenames = zip(*batch)
    max_width = max(img.shape[2] for img in images)
    padded_images = []
    for img in images:
        c, h, w = img.shape
        pad_w = max_width - w
        if pad_w > 0:
            padding = torch.zeros(c, h, pad_w)
            img = torch.cat([img, padding], dim=2)
        padded_images.append(img)
    images = torch.stack(padded_images, 0)
    return images, filenames

test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         collate_fn=collate_test, num_workers=NUM_WORKERS)

all_preds = []
all_filenames = []

with torch.no_grad():
    for images, batch_filenames in tqdm(test_loader, desc="Predicting"):
        images = images.to(DEVICE)
        
        # Получаем предсказания от всех моделей
        batch_preds_per_model = []
        for model in models:
            logits = model(images)
            pred_strings = decode_predictions(logits)
            batch_preds_per_model.append(pred_strings)
        
        # Транспонируем: для каждого примера собираем предсказания всех моделей
        for i, fname in enumerate(batch_filenames):
            model_preds = [preds[i] for preds in batch_preds_per_model]
            # Голосование: выбираем самую частую строку
            counter = Counter(model_preds)
            most_common = counter.most_common(1)[0][0]
            all_preds.append(most_common)
            all_filenames.append(fname)

submission = pd.DataFrame({'Filename': all_filenames, 'Price': all_preds})
submission.to_csv('submission_crnn_voting.csv', index=False)
print("Submission saved to submission_crnn_voting.csv")
print(submission.head())

Predicting: 100%|██████████| 12/12 [00:11<00:00,  1.00it/s]

Submission saved to submission_crnn_voting.csv
                                       Filename Price
0   8_D0-CF-13-24-59-DC_2026-01-22-14-02-47.jpg    34
1  12_10-B4-1D-E0-1E-60_2026-01-22-14-00-35.jpg   129
2   9_D0-CF-13-22-6F-AC_2026-01-22-14-03-36.jpg   109
3  15_10-B4-1D-C8-0C-D8_2026-01-22-14-02-31.jpg   199
4  28_D0-CF-13-24-CD-28_2026-01-22-18-28-43.jpg    89


In [48]:
# Проверка одной модели на одном изображении
test_img_path = os.path.join(TEST_IMG_DIR, sub_df.iloc[0]['Filename'])
print(f"Тестовое изображение: {test_img_path}")

# Загружаем и препроцессим как в датасете
image = cv2.imread(test_img_path)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
h, w = image.shape[:2]
new_w = int(w * (TARGET_HEIGHT / h))
image = cv2.resize(image, (new_w, TARGET_HEIGHT), interpolation=cv2.INTER_LINEAR)
augmented = test_transforms(image=image)
image_tensor = augmented['image'].unsqueeze(0).to(DEVICE)

print(f"Форма тензора: {image_tensor.shape}")

# Прогоняем через каждую модель
for i, model in enumerate(models):
    with torch.no_grad():
        logits = model(image_tensor)
        pred = decode_predictions(logits)[0]
        print(f"Модель {i}: {pred}")

Тестовое изображение: /home/shizm/DL_LABs/DL-lab4/dl-lab-4-ocr/test/test/8_D0-CF-13-24-59-DC_2026-01-22-14-02-47.jpg
Форма тензора: torch.Size([1, 3, 32, 48])
Модель 0: 34
Модель 1: 34
Модель 2: 34
Модель 3: 34
Модель 4: 34
Модель 5: 34
Модель 6: 34
Модель 7: 34
Модель 8: 34
Модель 9: 34


In [ ]:
Нужно делать голосование посимвольно!